In [1]:
#Import ibraries 
import sys
import re
import json
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fasttext

from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
sys.path.append(str(Path.cwd().parent / "src"))
from helpers import pick_by_pos, write_fasttext, train_fasttext, train_svm, eval_metrics_ft, eval_metrics_svm ,normalize_doc_id

#Set random seeds to ensure reproducibility of results
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

#Experiment parameters 
chunk_size = 2000
max_iteration = 30
TARGET_QUAL_AUTO = 2000
test_per_class = 20

# Dynamic confidence schedule
TAU_START = 0.70
TAU_END = 0.92
ramp_iteration = max_iteration

# Class imbalance controls
max_quant_to_qual_ratio= 1.5
max_quant_add_per_iteratioon= 200
undersample_quant_to_tual = 1.5

# Heuristic candidate review sample
heuristic_sample_frac = 0.03

cfg = {
    "epoch": 25,
    "lr": 0.5,
    "wordNgrams": 2,
    "dim": 100,
    "loss": "softmax",
    "minCount": 1,
    "minCountLabel": 0,
    "ws": 5,
    "thread": 4,
}
base_dir = Path.cwd().parent
data_dir = base_dir / "data"
output_dir = base_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)



In [2]:
# These lists represent manually identified document indices
# for qualitative and quantitative samples within the first 1000 records.


qual_doc_ids_1 = [85, 157, 203, 270, 277, 376, 481, 507, 512, 526, 529, 592, 609, 629,
                  659, 804, 891, 900, 911, 957, 976, 34, 75, 78, 79, 80, 118, 148, 213, 
                  215, 226, 438, 492, 514, 533, 569, 613, 614, 681, 746, 
                  747, 766, 808, 823, 897, 905, 907, 933, 944, 970]



quan_doc_ids_1 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 
                  21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 35, 36, 37, 38,
                  39, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 
                  57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 
                  74, 76, 77, 81, 82, 83, 84, 86, 87, 88, 89, 90, 91, 92, 93, 94, 
                  95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109,
                  110, 111, 112, 113, 114, 115, 116, 117, 119, 120, 121, 122, 123, 124, 
                  125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 
                  139, 140, 141, 142, 143, 145, 146, 147, 149, 150, 151, 152, 153,
                  154, 155, 156, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 
                  169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 
                  183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 
                  197, 198, 199, 200, 201, 202, 204, 205, 206, 207, 208, 209, 210, 211, 
                  212, 214, 217, 218, 219, 220, 221, 222, 223, 224, 225, 227, 228, 229, 
                  230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 
                  244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 257, 258, 
                  259, 260, 261, 262, 263, 265, 266, 267, 268, 269, 271, 272, 273, 274, 
                  275, 276, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289,
                  290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303,
                  304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 
                  318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 
                  332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 
                  346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 
                  360, 361, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 
                  375, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 
                  390, 391, 392, 393, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404,
                  405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 
                  419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 432, 433, 
                  434, 435, 436, 437, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 
                  450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 
                  464, 465, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 
                  479, 480, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 493, 494, 
                  495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 508, 509, 
                  510, 511, 513, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 
                  527, 528, 530, 531, 532, 534, 535, 536, 537, 538, 539, 540, 541, 542, 
                  543, 544, 545, 546, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 
                  558, 559, 560, 561, 562, 563, 564, 565, 566, 567, 568, 570, 571, 
                  572, 573, 574, 575, 576, 577, 578, 580, 581, 582, 583, 584, 585, 586, 
                  587, 588, 589, 590, 591, 593, 594, 595, 596, 597, 599, 600, 601, 602, 
                  603, 604, 605, 606, 607, 608, 610, 611, 612, 614, 615, 616, 617, 
                  618, 619, 620, 621, 622, 623, 624, 625, 626, 627, 628, 630, 631, 632, 
                  633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 
                  647, 648, 649, 650, 651, 652, 653, 655, 656, 657, 658, 660, 661, 662, 
                  663, 664, 665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 676, 677, 
                  678, 679, 680, 682, 683, 684, 685, 686, 687, 688, 689, 690, 691, 692,
                  693, 694, 695, 696, 697, 698, 699, 700, 701, 702, 704, 705, 706, 707, 
                  708, 709, 710, 711, 712, 713, 714, 715, 716, 717, 718, 719, 720, 721, 
                  722, 723, 724, 725, 726, 727, 728, 729, 730, 731, 732, 733, 734, 735, 
                  736, 737, 738, 739, 741, 742, 743, 744, 745, 748, 749, 750, 751, 
                  752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 
                  764, 765, 767, 768, 769, 771, 772, 773, 774, 775, 776, 777, 
                  778, 779, 780, 781, 782, 783, 784, 786, 787, 788, 789, 790, 791, 792, 
                  793, 794, 796, 797, 798, 799, 800, 801, 802, 803, 805, 806, 807,  
                  809, 810, 811, 812, 813, 814, 815, 816, 817, 818, 819, 820, 821, 822, 
                  824, 825, 826, 827, 828, 829, 830, 831, 832, 833, 834, 835, 836, 837, 
                  838, 839, 840, 841, 842, 843, 844, 845, 846, 847, 848, 849, 850, 851, 
                  853, 854, 855, 856, 857, 858, 859, 860, 861, 862, 863, 864, 865, 866, 
                  867, 868, 869, 870, 871, 872, 873, 874, 875, 876, 877, 878, 879, 880, 
                  881, 882, 884, 885, 886, 887, 888, 889, 890, 892, 893, 894, 895, 896, 
                  898, 901, 902, 903, 904, 906, 908, 909, 910, 912, 913, 914, 915, 916,
                  917, 918, 919, 920, 921, 922, 923, 924, 925, 926, 927, 928, 929, 930, 
                  931, 932, 934, 935, 936, 937, 938, 939, 940, 941, 942, 943, 945, 
                  946, 947, 948, 949, 950, 951, 952, 953, 954, 955, 956, 958, 959, 960, 
                  961, 962, 963, 964, 965, 966, 967, 968, 969, 971, 972, 973, 974, 975, 
                  977, 978, 979, 980, 981, 982, 983, 984, 985, 987, 988, 989, 990, 991, 
                  992, 993, 994, 995, 996, 997, 998, 999, 1000]

In [3]:
# Load the full dataset containing research abstracts
cs_df = pd.read_csv(data_dir / "data.csv")

In [4]:
# Define class labels
label_qual = "qualitative"
label_quant = "quantitative"

# Select first 1000 samples for controlled evaluation
first1000 = cs_df.head(1000).copy().reset_index(drop=True)
# Extract manually labeled qualitative and quantitative 
first1000_qual_eval = pick_by_pos(first1000, qual_doc_ids_1)
first1000_quant_eval = pick_by_pos(first1000, quan_doc_ids_1)

first1000_qual_eval["y_true"] = label_qual
first1000_quant_eval["y_true"] = label_quant
# Combine both classes into a single dataset
first1000_eval = pd.concat([first1000_qual_eval, first1000_quant_eval], ignore_index=True)
first1000_eval = first1000_eval.drop_duplicates(subset="doc_id", keep="first")

In [5]:
# Train fastText Baseline
train_txt = output_dir / "fasttext_train_seed.txt"
model_out = output_dir / "fasttext_train_seed.bin"
train_seed = pd.read_csv(output_dir / "train_seed_stage0.csv")
write_fasttext(train_seed, "abstract", "label", train_txt)
ft_model_seed = train_fasttext(train_txt, model_out, cfg)

Read 0M words
Number of words:  8252
Number of labels: 2
Progress: 100.0% words/sec/thread: 1106578 lr:  0.000000 avg.loss:  0.688513 ETA:   0h 0m 0s


In [6]:
#Build initial training pool and test data 
manual1_qual = pick_by_pos(first1000, qual_doc_ids_1)
manual1_quant = pick_by_pos(first1000, quan_doc_ids_1)

manual1_qual["label"] = label_qual
manual1_quant["label"] = label_quant

manual1 = pd.concat([manual1_qual, manual1_quant], ignore_index=True)
manual1 = manual1[["doc_id", "title", "abstract", "label"]].drop_duplicates(subset="doc_id", keep="first")

base_train_pool = pd.concat([
    train_seed[["doc_id", "title", "abstract", "label"]],
    manual1,
], ignore_index=True).drop_duplicates(subset="doc_id", keep="first")

base_train_pool = normalize_doc_id(base_train_pool)
base_train_pool["label_norm"] = base_train_pool["label"].astype(str).str.strip().str.lower()

# Save training pool
base_train_pool.to_csv(output_dir / "base_train_pool.csv", index=False)

#Save test data 
eval_df = test_df.copy() if "test_df" in globals() else pd.DataFrame(columns=["abstract", "y_true"])
eval_df.to_csv(output_dir / "eval_df.csv", index=False)

print("stage1 manual added:", len(manual1))
print("base_train_pool:", len(base_train_pool))
print(base_train_pool["label_norm"].value_counts())
print("eval_df:", eval_df.shape)

stage1 manual added: 975
base_train_pool: 1155
label_norm
quantitative    1005
qualitative      150
Name: count, dtype: int64
eval_df: (0, 2)


In [7]:
# Train SVM model using seed data
svm_model_seed = train_svm(train_seed)

In [8]:
# Evaluate FastText
test_df = pd.read_csv(output_dir / "test_df_stage0.csv")
ft_test_metrics = eval_metrics_ft(ft_model_seed, test_df, y_col="y_true")


In [9]:
# Evaluate SVM
svm_test_metrics = eval_metrics_svm(svm_model_seed, test_df, y_col="y_true")


In [10]:
#FastText evaluation on first 1000
ft_first1000_metrics = eval_metrics_ft(ft_model_seed, first1000_eval, y_col="y_true")


In [11]:
# SVM evaluation on first 100
svm_first1000_metrics = eval_metrics_svm(svm_model_seed, first1000_eval, y_col="y_true")


In [12]:
baseline_results = pd.DataFrame([
    {"model": "fasttext", "dataset": "test_df", **ft_test_metrics},
    {"model": "svm", "dataset": "test_df", **svm_test_metrics},
    {"model": "fasttext", "dataset": "first1000_eval", **ft_first1000_metrics},
    {"model": "svm", "dataset": "first1000_eval", **svm_first1000_metrics},
])



In [13]:
baseline_results

,model,dataset,acc,f1_macro,f1_qual,f1_quant,precision_macro,recall_macro
0,fasttext,test_df,0.625000,0.605003,0.516129,0.693878,0.656740,0.625000
1,svm,test_df,0.850000,0.849624,0.842105,0.857143,0.853535,0.850000
2,fasttext,first1000_eval,0.909744,0.670081,0.388889,0.951274,0.636450,0.744324
3,svm,first1000_eval,0.725128,0.527028,0.220930,0.833126,0.555815,0.741622
